In [1]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import make_scorer, accuracy_score
import numpy as np
import xgboost as xgb
import json
import pandas as pd
from pandas import json_normalize
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from textblob import TextBlob
import re
from sklearn.ensemble import RandomForestClassifier
from gensim.models import Word2Vec
from gensim.models import Word2Vec


In [2]:
# ===============================
# JSONL LOADING
# ===============================


# Load the training data from a JSON Lines file (one JSON object per line)
train_data = pd.read_json('train.jsonl', lines=True)
# The tweet data is nested. json_normalize flattens the nested JSON into columns.
train_data = json_normalize(train_data.to_dict(orient='records'))

# Load the Kaggle test data (which we will make predictions on)
kaggle_data = pd.read_json('kaggle_test.jsonl', lines=True)
# Also normalize the Kaggle data
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))


# Separate features from the target variable for the training set
X_train = train_data.drop('label', axis=1)
y_train = train_data['label']

X_kaggle = kaggle_data

print("✓ Chargement OK")

✓ Chargement OK


In [3]:
import pandas as pd
import numpy as np
from datetime import datetime

def create_advanced_features(df_input):
    df = df_input.copy()
    
    # Définition des séries de fallback (robustesse contre les colonnes manquantes)
    default_int_series = pd.Series(0, index=df.index)
    default_bool_series = pd.Series(False, index=df.index)
    
    # --- Initialisation des colonnes numériques ---
    df['user.followers_count'] = df.get('user.followers_count', default_int_series).fillna(0)
    df['user.friends_count'] = df.get('user.friends_count', default_int_series).fillna(0)
    df['user.listed_count'] = df.get('user.listed_count', default_int_series).fillna(0)
    df['user.favourites_count'] = df.get('user.favourites_count', default_int_series).fillna(0)
    df['user.statuses_count'] = df.get('user.statuses_count', default_int_series).fillna(0)
    df['retweet_count'] = df.get('retweet_count', default_int_series).fillna(0)
    df['favorite_count'] = df.get('favorite_count', default_int_series).fillna(0)
    df['quote_count'] = df.get('quote_count', default_int_series).fillna(0) # Nouveau : Quote Count
    df['reply_count'] = df.get('reply_count', default_int_series).fillna(0) # Nouveau : Reply Count
    
    # --- A. GESTION DES DATES (Ancienneté du compte) ---
    df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')
    ref_date = pd.to_datetime('now', utc=True)
    df['account_age_days'] = (ref_date - df['user_created_at_dt']).dt.days
    df['account_age_days'] = df['account_age_days'].fillna(0)
    
    # Extraction des indicateurs temporels avancés du tweet
    df['created_at_dt'] = pd.to_datetime(df.get('created_at'), errors='coerce')
    df['tweet_hour'] = df['created_at_dt'].dt.hour.fillna(-1)
    df['tweet_is_weekend'] = df['created_at_dt'].dt.dayofweek.isin([5, 6]).fillna(False).astype(int)

    # --- B. QUALITÉ DU PROFIL & STATUT (Booléens) ---
    df['is_default_profile'] = df.get('user.default_profile', default_bool_series).fillna(False).astype(int)
    df['is_default_image'] = df.get('user.default_profile_image', default_bool_series).fillna(False).astype(int)
    df['is_verified'] = df.get('user.verified', default_bool_series).fillna(False).astype(int)
    
    # Nouveau : Est-ce un compte protégé/privé ? (Signe d'un follower ou d'un utilisateur personnel)
    df['is_protected'] = df.get('user.protected', default_bool_series).fillna(False).astype(int)
    
    # Nouveau : Le profil a-t-il une URL renseignée ?
    df['has_url'] = df.get('user.url', pd.Series(False, index=df.index)).notna().astype(int)

    # --- C. CONTENU DU TWEET (Entities Counting) ---
    def count_entities(x):
        if isinstance(x, list) or (isinstance(x, pd.Series) and x.dtype == object): return len(x)
        return 0

    df['num_urls'] = df.get('entities.urls', default_int_series).apply(count_entities)
    df['num_hashtags'] = df.get('entities.hashtags', default_int_series).apply(count_entities)
    df['num_mentions'] = df.get('entities.user_mentions', default_int_series).apply(count_entities)
    df['has_media'] = df.get('extended_entities.media', default_bool_series).notna().astype(int)

    # --- D. RATIOS PUISSANTS & COMPORTEMENTAUX ---
    followers = df['user.followers_count']
    friends = df['user.friends_count']
    listed = df['user.listed_count']
    statuses = df['user.statuses_count']
    
    # 1. Ratio Followers / Friends (Ratio de notoriété)
    df['ratio_followers_friends'] = followers / (friends + 1)
    df['ratio_listed_followers'] = listed / (followers + 1)
    
    # 2. Taux de Réciprocité d'Amitié (Nouveau - Indique un équilibre/déséquilibre d'influence)
    df['reciprocity_score'] = (friends - followers) / (friends + followers + 1)

    # 3. Activité (Tweets par jour d'existence)
    df['tweets_per_day'] = statuses / (df['account_age_days'] + 1)
    
    # 4. Ratio Mention/Tweet (Nouveau - Taux d'interaction vs. diffusion)
    # Plus ce ratio est élevé, plus l'utilisateur interagit personnellement (follower).
    df['ratio_mention_status'] = df['num_mentions'] / (statuses + 1)
    
    # 5. Engagement (Taux d'engagement par Tweet)
    total_engagement = df['retweet_count'] + df['favorite_count'] + df['quote_count'] + df['reply_count']
    df['total_tweet_engagement'] = total_engagement / (followers + 1)

    # --- E. LONGUEUR DES TEXTES ---
    df['final_text'] = df.get('extended_tweet.full_text', df.get('text', pd.Series(''))).fillna('')
    df['final_text'] = df['final_text'].where(df['final_text'] != '', df.get('text', '')).fillna('')
    
    df['text_length'] = df['final_text'].astype(str).apply(len)
    df['bio_length'] = df.get('user.description', '').astype(str).apply(len)

    # --- F. SÉLECTION FINALE ---
    features_to_keep = [
        # Métriques User Brutes
        'user.followers_count', 'user.friends_count', 'user.listed_count', 
        'user.favourites_count', 'user.statuses_count',
        # Métriques Tweet Brutes
        'retweet_count', 'favorite_count', 'quote_count', 'reply_count',
        # Ratios & Comportementaux (NOUVEAU)
        'ratio_followers_friends', 'ratio_listed_followers', 'tweets_per_day', 'account_age_days',
        'reciprocity_score', 'ratio_mention_status', 'total_tweet_engagement',
        # Booléens & Qualité (MIS À JOUR)
        'is_verified', 'is_default_profile', 'is_default_image', 'is_geo_enabled',
        'is_protected', 'has_url',
        # Temporels (NOUVEAU)
        'tweet_hour', 'tweet_is_weekend',
        # Longueur du Contenu
        'text_length', 'bio_length', 
        # Compte des Entités
        'num_urls', 'num_hashtags', 'num_mentions', 'has_media',
    ]
    
    final_cols = [c for c in features_to_keep if c in df.columns]
    
    return df[final_cols].fillna(0)



In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from textblob import TextBlob
import re
from nltk.corpus import stopwords

def create_nlp_features(df_train, df_test, y_train):
    """
    Crée des features NLP (TF-IDF des Bios et Sentiment des Tweets).

    Args:
        df_train (pd.DataFrame): DataFrame d'entraînement complet (X_train).
        df_test (pd.DataFrame): DataFrame de test (X_kaggle).
        y_train (pd.Series): Cible d'entraînement (y_train).

    Returns:
        tuple: (df_train_nlp, df_test_nlp) avec les nouvelles colonnes.
    """
    
    # ----------------------------------------
    # Préparation du texte
    # ----------------------------------------

    french_stopwords = stopwords.words("french")
    
    # Remplacer les NaN ou valeurs manquantes par une chaîne vide
    train_bio = df_train.get('user.description', pd.Series([''] * len(df_train))).fillna('').astype(str)
    test_bio = df_test.get('user.description', pd.Series([''] * len(df_test))).fillna('').astype(str)

    # Récupération du 'final_text' du tweet (le plus complet)
    # Note: On doit reproduire la logique de 'final_text' de la fonction d'ingénierie
    def get_final_text(df):
        text = df.get('text', pd.Series([''] * len(df))).fillna('')
        full_text = df.get('extended_tweet.full_text', text).fillna(text)
        return full_text.astype(str)
        
    train_text = get_final_text(df_train)
    test_text = get_final_text(df_test)

    # ----------------------------------------
    # A. TF-IDF sur les tweets (Meta-Feature)
    # ----------------------------------------
    print("TF-IDF vectorization du corps du tweet")

    # Nettoyage très basique du texte pour le TF-IDF
    def clean_text(text):
        #text = text.lower()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Suppression des URLs
        #text = re.sub(r'[^\w\s]', '', text) # Suppression de la ponctuation
        return text

    train_text_clean = train_text.apply(clean_text)
    test_text_clean = test_text.apply(clean_text)

    # 1. Création du Vecteur TF-IDF (Appris uniquement sur le training set)
    tfidf = TfidfVectorizer(max_features=1000, stop_words=french_stopwords, ngram_range=(2, 5), analyzer='char_wb', lowercase=False)
    X_train_tfidf_tweet = tfidf.fit_transform(train_text_clean)
    X_test_tfidf_tweet = tfidf.transform(test_text_clean)

    # 2. Entraînement du Méta-Modèle (Régression Logistique)
    log_reg = LogisticRegression(solver='sag', random_state=42)
    log_reg.fit(X_train_tfidf_tweet, y_train.astype(int))

    # 3. Extraction de la probabilité prédite (Meta-Feature)
    # Nous utilisons la probabilité pour la classe 1 (Influencer)
    train_tweet_proba = log_reg.predict_proba(X_train_tfidf_tweet)[:, 1]
    test_tweet_proba = log_reg.predict_proba(X_test_tfidf_tweet)[:, 1]    

    # ----------------------------------------
    # B. TF-IDF sur les Bios (Meta-Feature)
    # ----------------------------------------
    print("  -> Calcul du TF-IDF sur les Bios et entraînement du Méta-Modèle...")
    

    train_bio_clean = train_bio.apply(clean_text)
    test_bio_clean = test_bio.apply(clean_text)

    # 1. Création du Vecteur TF-IDF (Appris uniquement sur le training set)
    tfidf = TfidfVectorizer(max_features=1000, stop_words=french_stopwords, ngram_range=(2, 5), analyzer='char_wb', lowercase=False)
    X_train_tfidf = tfidf.fit_transform(train_bio_clean)
    X_test_tfidf = tfidf.transform(test_bio_clean)

    # 2. Entraînement du Méta-Modèle (Régression Logistique)
    log_reg = LogisticRegression(solver='liblinear', random_state=42)
    log_reg.fit(X_train_tfidf, y_train.astype(int))

    # 3. Extraction de la probabilité prédite (Meta-Feature)
    # Nous utilisons la probabilité pour la classe 1 (Influencer)
    train_bio_proba = log_reg.predict_proba(X_train_tfidf)[:, 1]
    test_bio_proba = log_reg.predict_proba(X_test_tfidf)[:, 1]

    # ----------------------------------------
    # B. Analyse du Sentiment (Polarity et Subjectivity)
    # ----------------------------------------
    print("  -> Extraction du Sentiment (Polarity/Subjectivity) des Tweets...")
    
    # La fonction TextBlob est utilisée pour obtenir les scores de sentiment
    # C'est une opération lente, il faut être patient
    def get_sentiment(text):
        try:
            analysis = TextBlob(text)
            return pd.Series({'polarity': analysis.sentiment.polarity, 'subjectivity': analysis.sentiment.subjectivity})
        except:
            return pd.Series({'polarity': 0.0, 'subjectivity': 0.0})

    train_sentiment = train_text.apply(get_sentiment)
    test_sentiment = test_text.apply(get_sentiment)

    # ----------------------------------------
    # 4. Fusion des Features NLP
    # ----------------------------------------
    
    # Création des DataFrames de features NLP
    df_train_nlp = pd.DataFrame({
        'meta_bio_proba': train_bio_proba,
        'tweet_polarity': train_sentiment['polarity'],
        'tweet_subjectivity': train_sentiment['subjectivity'],
        'meta_tweet_proba': train_tweet_proba
    })
    
    df_test_nlp = pd.DataFrame({
        'meta_bio_proba': test_bio_proba,
        'tweet_polarity': test_sentiment['polarity'],
        'tweet_subjectivity': test_sentiment['subjectivity'],
        'meta_tweet_proba': test_tweet_proba
    })

    return df_train_nlp, df_test_nlp

In [5]:
# =======================================================
# On concatène ca dans X_train_advanced et X_kaggle_advanced
# =======================================================
print("🛠️ Construction des features avancées (Métadonnées)...")

# Application de la fonction robuste (Métadonnées)
X_train_metadata = create_advanced_features(X_train)
X_kaggle_metadata = create_advanced_features(X_kaggle)

# Préparation de la cible
y_train_clean = y_train.astype(int)

# --- NOUVELLE ÉTAPE : CRÉATION DES FEATURES NLP ---
X_train_nlp, X_kaggle_nlp = create_nlp_features(X_train, X_kaggle, y_train_clean)

# # --- FUSION DES FEATURES ---
# X_train_advanced = pd.concat([X_train_advanced, X_train_nlp], axis=1)
# X_kaggle_advanced = pd.concat([X_kaggle_advanced, X_kaggle_nlp], axis=1)

# print(f"\nFeatures combinées ({len(X_train_advanced.columns)}):")
# print(list(X_train_advanced.columns))

🛠️ Construction des features avancées (Métadonnées)...


/tmp/ipykernel_525/3318943902.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')
/tmp/ipykernel_525/3318943902.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')


TF-IDF vectorization du corps du tweet


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:539: UserWarning: The parameter 'stop_words' will not be used since 'analyzer' != 'word'
  warnings.warn(


  -> Calcul du TF-IDF sur les Bios et entraînement du Méta-Modèle...


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:539: UserWarning: The parameter 'stop_words' will not be used since 'analyzer' != 'word'
  warnings.warn(


  -> Extraction du Sentiment (Polarity/Subjectivity) des Tweets...


On ajoute les embedding par Word2vec

In [6]:
def simple_tokenize(text):
    """
    Tokenise un texte en le mettant en minuscules et en le séparant par espace.
    Nettoie les caractères non alphanumériques courants pour Word2Vec.
    """
    # 1. Mise en minuscule
    text = text.lower()
    
    # 2. Remplacer la ponctuation courante par un espace pour la séparation
    # Ceci est une simplification pour Word2Vec
    text = re.sub(r'[.,;:`"\'!?()]', ' ', text)
    
    # 3. Séparation par espace et suppression des entrées vides
    return [word for word in text.split() if word]

# ====================================================================
# 2. CHARGEMENT ET PRÉPARATION DES DONNÉES (Votre Bloc)
# ====================================================================

# --- 1. FONCTION D'EXTRACTION DE TEXTE ---
def extract_full_text(row):
    """Extrait le texte le plus complet disponible du tweet."""
    if "extended_tweet.full_text" in row and pd.notna(row["extended_tweet.full_text"]):
        return row["extended_tweet.full_text"]
    if "text" in row and pd.notna(row["text"]):
        return row["text"]
    return ""

# --- 2. CHARGEMENT DES DONNÉES BRUTES ---
try:
    # 🚨 Adaptez les chemins de fichiers si nécessaire !
    # Application de l'extraction de texte
    train_data["full_text"] = train_data.apply(extract_full_text, axis=1)
    kaggle_data["full_text"] = kaggle_data.apply(extract_full_text, axis=1)

    # DÉFINITION DES VARIABLES POUR LE PIPELINE W2V
    X_full = train_data["full_text"].values # Texte complet d'entraînement
    y_full = train_data["label"].values.astype(int) # Labels
    X_kaggle_full = kaggle_data["full_text"].values # Texte complet Kaggle
    
    print("✓ Données brutes (X_full, y_full) chargées et prêtes.")

except FileNotFoundError as e:
    print(f"❌ ERREUR: Fichier introuvable. Vérifiez votre chemin : {e}")
    exit()

✓ Données brutes (X_full, y_full) chargées et prêtes.


In [7]:
# ====================================================================
# CONFIGURATION WORD2VEC (Assurez-vous que ces valeurs sont correctes)
# ====================================================================
EMBEDDING_DIM = 250 
WINDOW_SIZE = 5
MIN_COUNT = 1
# Note : Si vous avez entraîné votre modèle W2V sur toutes les données, 
# vous pouvez utiliser X_full. Sinon, utilisez X_train_text.
# Ici, nous utilisons les ensembles complets pour la vectorisation.

# ====================================================================
# 1. PRÉPARATION DES DONNÉES DE TEXTE (Tokenisation)
# ====================================================================

# 🚨 Assurez-vous que X_full et X_kaggle_full contiennent le texte brut !

# J'utilise ici la fonction simple_tokenize définie dans la réponse précédente
# (Elle doit être définie dans votre session pour que ce bloc fonctionne)

print("\n🔄 Tokenisation des données de texte...")
# 1. Tokenisation du jeu d'entraînement (X_full)
X_full_tokenized = [simple_tokenize(text) for text in X_full]

# 2. Tokenisation du jeu Kaggle (X_kaggle_full)
X_kaggle_tokenized = [simple_tokenize(text) for text in X_kaggle_full]


# ====================================================================
# 2. FONCTION DE VECTORISATION
# ====================================================================

def document_vectorizer(tokens, model, dim):
    """Calcule le vecteur moyen pour un document."""
    vector = np.zeros(dim)
    count = 0
    # Ne fait la moyenne que sur les mots présents dans le vocabulaire W2V
    for word in tokens:
        if word in model.wv:
            vector += model.wv[word]
            count += 1
    
    if count != 0:
        vector /= count
        
    return vector

# ====================================================================
# 3. GÉNÉRATION DES EMBEDDINGS (VECTEURS)
# ====================================================================

print("🔄 Génération des embeddings W2V par moyenne...")

# 🚨 Le modèle w2v_model doit être défini ou chargé ici
# Si vous ne l'avez pas entraîné ou chargé dans la même session, vous devez le faire ici !
# Exemple: w2v_model = Word2Vec.load("mon_modele_w2v.model")

w2v_model = Word2Vec(
    sentences=X_full_tokenized, 
    vector_size=EMBEDDING_DIM, 
    window=WINDOW_SIZE, 
    min_count=MIN_COUNT, 
    sg=1 # 1: Skip-gram (souvent meilleur pour la sémantique), 0: CBOW
)

# 3.1. Entraînement
X_train_vectors = np.array([document_vectorizer(tokens, w2v_model, EMBEDDING_DIM) for tokens in X_full_tokenized])

# 3.2. Kaggle
X_kaggle_vectors = np.array([document_vectorizer(tokens, w2v_model, EMBEDDING_DIM) for tokens in X_kaggle_tokenized])


# ====================================================================
# 4. FUSION DES EMBEDDINGS DANS X_train_advanced
# ====================================================================

print("🔄 Fusion des embeddings W2V avec les jeux X_advanced...")

# Création des DataFrames d'Embeddings
embed_cols = [f'w2v_e_{i}' for i in range(EMBEDDING_DIM)]

X_train_nlp = pd.DataFrame(X_train_vectors, columns=embed_cols)
X_kaggle_nlp = pd.DataFrame(X_kaggle_vectors, columns=embed_cols)

# # Fusion des Caractéristiques (Métadonnées + Embeddings W2V)
# X_train_advanced = pd.concat([X_train_advanced, X_train_nlp], axis=1)
# X_kaggle_advanced = pd.concat([X_kaggle_advanced, X_kaggle_nlp], axis=1)

# print("\n" + "=" * 50)
# print("✓ FUSION DES EMBEDDINGS W2V TERMINÉE")
# print("=" * 50)
# print(f"Total Features d'entraînement (X_train_advanced) : {len(X_train_advanced.columns)}")
# print(f"Total Features de test (X_kaggle_advanced) : {len(X_kaggle_advanced.columns)}")


🔄 Tokenisation des données de texte...
🔄 Génération des embeddings W2V par moyenne...
🔄 Fusion des embeddings W2V avec les jeux X_advanced...


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset


In [10]:
# Features and labels
X_train_metadata_tensor = torch.tensor(X_train_metadata.values, dtype=torch.float32)
X_train_nlp_tensor = torch.tensor(X_train_nlp.values, dtype=torch.float32)

X_kaggle_metadata_tensor = torch.tensor(X_kaggle_metadata.values, dtype=torch.float32)
X_kaggle_nlp_tensor = torch.tensor(X_kaggle_nlp.values, dtype=torch.float32)


y_train_tensor = torch.tensor(y_train_clean.values, dtype=torch.long)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_train_metadata_tensor = X_train_metadata_tensor.to(device)
X_train_nlp_tensor = X_train_nlp_tensor.to(device)
X_kaggle_metadata_tensor = X_kaggle_metadata_tensor.to(device)
X_kaggle_nlp_tensor = X_kaggle_nlp_tensor.to(device)
y_train_tensor = y_train_tensor.to(device)


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TabTransformerLite(nn.Module):
    def __init__(self, input_dim, emb_dim=32, num_heads=4, num_layers=3, dropout=0.1):
        super().__init__()

        # Embedding pour chaque feature scalaire : (1 → emb_dim)
        self.feature_embedding = nn.Linear(1, emb_dim)

        # Normalisation après embedding
        self.pre_norm = nn.LayerNorm(emb_dim)

        # Transformer Encoder très léger
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=num_heads,
            dim_feedforward=emb_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # # Mean pooling : résume toutes les features
        # # (pas de flatten -> modèle minuscule)
        # self.classifier = nn.Sequential(
        #     nn.Linear(emb_dim, 64),
        #     nn.ReLU(),
        #     # nn.Dropout(dropout),
        #     # nn.Linear(64, 2)
        # )

    def forward(self, x):
        # x shape: (batch, input_dim)
        x = x.unsqueeze(-1)                         # (batch, features, 1)
        x = self.feature_embedding(x)               # (batch, features, emb_dim)
        x = self.pre_norm(x)
        x = self.transformer(x)                     # encode features
        x = x.mean(dim=1)                           # pooling (batch, emb_dim)
        return x


class MetaEncoder(nn.Module):
    def __init__(self, input_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)                  # (batch, hidden)


class FusionClassifier(nn.Module):
    def __init__(self, text_dim, meta_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(text_dim + meta_dim, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)


class UnifiedModel(nn.Module):
    def __init__(self, vocab_size, meta_dim, num_classes,
                 text_emb_dim=128, text_heads=4, text_layers=2):
        super().__init__()

        self.text_encoder = TabTransformerLite(
            input_dim=vocab_size,
            emb_dim=text_emb_dim,
            num_heads=text_heads,
            num_layers=text_layers
        )

        self.meta_encoder = MetaEncoder(
            input_dim=meta_dim,
            hidden=128
        )

        self.classifier = FusionClassifier(
            text_dim=text_emb_dim,
            meta_dim=128,
            num_classes=num_classes
        )

    def forward(self, tokens, metadata):
        h_text = self.text_encoder(tokens)    # (batch, T)
        h_meta = self.meta_encoder(metadata)  # (batch, M)

        h = torch.cat([h_text, h_meta], dim=1)

        return self.classifier(h)


In [51]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UnifiedModel(
    vocab_size=X_train_nlp.shape[1],
    meta_dim=X_train_metadata.shape[1],
    num_classes=2,
    text_emb_dim=256,
    text_heads=16,
    text_layers=10
).to(device)


In [52]:
train_dataset_metadata = TensorDataset(X_train_metadata_tensor, y_train_tensor)
train_dataset_nlp = TensorDataset(X_train_nlp_tensor, y_train_tensor)
loader_metadata = DataLoader(train_dataset_metadata, batch_size=64, shuffle=True)
loader_nlp = DataLoader(train_dataset_nlp, batch_size=64, shuffle=True)
loader_metadata.__dict__

{'dataset': <torch.utils.data.dataset.TensorDataset at 0x7fe7d042c560>,
 'num_workers': 0,
 'prefetch_factor': None,
 'pin_memory': False,
 'pin_memory_device': '',
 'timeout': 0,
 'worker_init_fn': None,
 '_DataLoader__multiprocessing_context': None,
 'in_order': True,
 '_dataset_kind': 0,
 'batch_size': 64,
 'drop_last': False,
 'sampler': <torch.utils.data.sampler.RandomSampler at 0x7fe7d06cc890>,
 'batch_sampler': <torch.utils.data.sampler.BatchSampler at 0x7fe7d06cc4a0>,
 'generator': None,
 'collate_fn': <function torch.utils.data._utils.collate.default_collate(batch)>,
 'persistent_workers': False,
 '_DataLoader__initialized': True,
 '_IterableDataset_len_called': None,
 '_iterator': None}

In [53]:
def predict_in_batches(model, X_nlp, X_metadata, batch_size=256):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_nlp), batch_size):
            xbnlp = X_nlp[i:i+batch_size].to(device)
            xbmeta = X_metadata[i:i+batch_size].to(device)
            pb = model(xbnlp, xbmeta).cpu()
            preds.append(pb)
    return torch.cat(preds, dim=0)

In [54]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
epochs = 50

patience = 5
patience_counter = 0
best_acc = 0.0
best_state = None

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for (xbnlp, yb), (xbmeta, yb) in zip(loader_nlp, loader_metadata):
        #xb = xb.to(device)
        #yb = yb.to(device)

        optimizer.zero_grad()
        out = model(xbnlp, xbmeta)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Accuracy train
    model.eval()
    with torch.no_grad():
        preds = predict_in_batches(model, X_train_nlp_tensor, X_train_metadata_tensor)
        preds = preds.to(device)
        acc = (preds.argmax(1) == y_train_tensor).float().mean().item()

    print(f"[Epoch {epoch+1}] Loss={total_loss/len(loader_nlp):.4f} | Acc={acc:.4f}")

    # Early stopping
    if acc > best_acc:
        best_acc = acc
        best_state = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("⛔ Early stopping triggered")
        break

model.load_state_dict(best_state)


[Epoch 1] Loss=2.3959 | Acc=0.7616
[Epoch 2] Loss=1.1578 | Acc=0.7631
[Epoch 3] Loss=0.8664 | Acc=0.7285
[Epoch 4] Loss=0.6616 | Acc=0.7614
[Epoch 5] Loss=0.5475 | Acc=0.7899
[Epoch 6] Loss=0.4819 | Acc=0.7881
[Epoch 7] Loss=0.4594 | Acc=0.7995
[Epoch 8] Loss=0.4431 | Acc=0.8056
[Epoch 9] Loss=0.4373 | Acc=0.8073
[Epoch 10] Loss=0.4337 | Acc=0.8083
[Epoch 11] Loss=0.4309 | Acc=0.8078
[Epoch 12] Loss=0.4282 | Acc=0.8070
[Epoch 13] Loss=0.4250 | Acc=0.8128
[Epoch 14] Loss=0.4248 | Acc=0.8052
[Epoch 15] Loss=0.4218 | Acc=0.8107
[Epoch 16] Loss=0.4221 | Acc=0.8140
[Epoch 17] Loss=0.4195 | Acc=0.8150
[Epoch 18] Loss=0.4188 | Acc=0.8027
[Epoch 19] Loss=0.4184 | Acc=0.8164
[Epoch 20] Loss=0.4157 | Acc=0.8139
[Epoch 21] Loss=0.4146 | Acc=0.8052
[Epoch 22] Loss=0.4132 | Acc=0.8163
[Epoch 23] Loss=0.4132 | Acc=0.8180
[Epoch 24] Loss=0.4126 | Acc=0.8126
[Epoch 25] Loss=0.4112 | Acc=0.8178
[Epoch 26] Loss=0.4103 | Acc=0.8138
[Epoch 27] Loss=0.4089 | Acc=0.8206
[Epoch 28] Loss=0.4091 | Acc=0.8207
[

<All keys matched successfully>

In [55]:
torch.save(model.state_dict(), "model_weights.pth")

In [56]:
model_cpu = model.to("cpu")

batch_size = 2048
preds = []

model_cpu.eval()
with torch.no_grad():
    for i in range(0, len(X_kaggle_metadata_tensor), batch_size):
        batch_nlp = X_kaggle_nlp_tensor[i:i+batch_size].to("cpu")
        batch_metadata = X_kaggle_metadata_tensor[i:i+batch_size].to("cpu")
        out = model_cpu(batch_nlp, batch_metadata)
        preds.append(out.argmax(1).numpy())

y_pred_kaggle = np.concatenate(preds)


In [57]:
# Save submission
output_dl = pd.DataFrame({
    'ID': X_kaggle['challenge_id'],
    'Prediction': y_pred_kaggle
})
# output_dl.to_csv('submission_transfo_lite.csv', index=False)
# print("✓ Submission for PyTorch model saved as 'submission_transfo_lite.csv'")

In [58]:
output_dl.to_csv('submission_unified.csv', index=False)
print("✓ Submission for PyTorch model saved as 'submission_unified.csv'")

✓ Submission for PyTorch model saved as 'submission_unified.csv'
